# CrossTL PBR Showcase

This notebook translates one CrossGL PBR shader into multiple GPU targets from the local checkout. It intentionally avoids installing `crosstl` from PyPI so the generated output reflects the code in this repository.


In [1]:
from pathlib import Path
import sys
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "crosstl").is_dir() and (candidate / "setup.py").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the CrossTL repository checkout.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import crosstl

SOURCE_PATH = REPO_ROOT / "demos" / "readme" / "showcase_pbr.cgl"
import_path = Path(crosstl.__file__).resolve()
try:
    import_display = import_path.relative_to(REPO_ROOT)
except ValueError:
    import_display = import_path

print("Repository root: local checkout")
print(f"CrossTL import:  {import_display}")
print(f"Shader source:   {SOURCE_PATH.relative_to(REPO_ROOT)}")


Repository root: local checkout
CrossTL import:  crosstl/__init__.py
Shader source:   demos/readme/showcase_pbr.cgl


## Source Shader

The demo source uses the current CrossGL struct-return stage style. That shape gives graphics backends explicit input/output structs and avoids stale global-stage syntax.

In [2]:
source_code = SOURCE_PATH.read_text(encoding="utf-8")
assert "shader ShowcasePBRShader" in source_code
assert "VertexOutput main(VertexInput input)" in source_code
assert "FragmentOutput main(FragmentInput input)" in source_code
assert "texture(albedoMap, input.uv)" in source_code

display(Markdown(f"""```cpp
{source_code}
```"""))


```cpp
shader ShowcasePBRShader {
    const float PI = 3.14159265359;
    const float EPSILON = 0.0001;
    const int MAX_LIGHTS = 4;

    struct VertexInput {
        vec3 position;
        vec3 normal;
        vec3 tangent;
        vec2 texCoord;
    }

    struct VertexOutput {
        vec3 worldPosition;
        vec3 worldNormal;
        vec3 worldTangent;
        vec3 worldBitangent;
        vec2 uv;
        vec4 position;
    }

    struct FragmentInput {
        vec3 worldPosition;
        vec3 worldNormal;
        vec3 worldTangent;
        vec3 worldBitangent;
        vec2 uv;
    }

    struct FragmentOutput {
        vec4 color;
    }

    float distributionGGX(vec3 normal, vec3 halfVector, float roughness) {
        float alpha = roughness * roughness;
        float alpha2 = alpha * alpha;
        float ndoth = max(dot(normal, halfVector), 0.0);
        float denom = ndoth * ndoth * (alpha2 - 1.0) + 1.0;
        return alpha2 / max(PI * denom * denom, EPSILON);
    }

    float geometrySchlickGGX(float ndotv, float roughness) {
        float k = (roughness + 1.0) * (roughness + 1.0) / 8.0;
        return ndotv / max(ndotv * (1.0 - k) + k, EPSILON);
    }

    float geometrySmith(vec3 normal, vec3 viewDir, vec3 lightDir, float roughness) {
        float ndotv = max(dot(normal, viewDir), 0.0);
        float ndotl = max(dot(normal, lightDir), 0.0);
        return geometrySchlickGGX(ndotv, roughness) * geometrySchlickGGX(ndotl, roughness);
    }

    vec3 fresnelSchlick(float cosTheta, vec3 baseReflectance) {
        return baseReflectance + (vec3(1.0) - baseReflectance) * pow(max(1.0 - cosTheta, 0.0), 5.0);
    }

    vertex {
        uniform mat4 modelMatrix;
        uniform mat4 viewProjectionMatrix;
        uniform mat3 normalMatrix;

        VertexOutput main(VertexInput input) {
            VertexOutput output;
            vec4 worldPosition = modelMatrix * vec4(input.position, 1.0);
            output.worldPosition = worldPosition.xyz;
            output.worldNormal = normalize(normalMatrix * input.normal);
            output.worldTangent = normalize(normalMatrix * input.tangent);
            output.worldBitangent = normalize(cross(output.worldNormal, output.worldTangent));
            output.uv = input.texCoord;
            output.position = viewProjectionMatrix * worldPosition;
            return output;
        }
    }

    fragment {
        uniform sampler2D albedoMap;
        uniform sampler2D normalMap;
        uniform sampler2D metallicRoughnessMap;
        uniform vec3 cameraPosition;
        uniform vec3 lightPositions[MAX_LIGHTS];
        uniform vec3 lightColors[MAX_LIGHTS];
        uniform int activeLightCount;
        uniform vec3 baseColor;
        uniform float metallicFactor;
        uniform float roughnessFactor;

        vec3 decodeNormal(vec3 encodedNormal, mat3 tangentFrame) {
            vec3 tangentNormal = encodedNormal * 2.0 - 1.0;
            return normalize(tangentFrame * tangentNormal);
        }

        FragmentOutput main(FragmentInput input) {
            FragmentOutput output;
            mat3 tangentFrame = mat3(
                normalize(input.worldTangent),
                normalize(input.worldBitangent),
                normalize(input.worldNormal)
            );
            vec3 normal = decodeNormal(texture(normalMap, input.uv).rgb, tangentFrame);
            vec3 viewDir = normalize(cameraPosition - input.worldPosition);
            vec3 albedo = texture(albedoMap, input.uv).rgb * baseColor;
            vec3 materialSample = texture(metallicRoughnessMap, input.uv).rgb;
            float metallic = clamp(materialSample.b * metallicFactor, 0.0, 1.0);
            float roughness = clamp(materialSample.g * roughnessFactor, 0.04, 1.0);
            vec3 baseReflectance = mix(vec3(0.04), albedo, metallic);
            vec3 radianceSum = vec3(0.0);

            for (int i = 0; i < MAX_LIGHTS; i++) {
                if (i >= activeLightCount) {
                    break;
                }
                vec3 lightVector = lightPositions[i] - input.worldPosition;
                float distanceSq = max(dot(lightVector, lightVector), EPSILON);
                vec3 lightDir = normalize(lightVector);
                vec3 halfVector = normalize(viewDir + lightDir);
                float attenuation = 1.0 / distanceSq;
                vec3 radiance = lightColors[i] * attenuation;

                float normalDistribution = distributionGGX(normal, halfVector, roughness);
                float geometry = geometrySmith(normal, viewDir, lightDir, roughness);
                vec3 fresnel = fresnelSchlick(max(dot(halfVector, viewDir), 0.0), baseReflectance);
                vec3 diffuseShare = (vec3(1.0) - fresnel) * (1.0 - metallic);
                vec3 numerator = normalDistribution * geometry * fresnel;
                float denominator = max(4.0 * max(dot(normal, viewDir), 0.0) * max(dot(normal, lightDir), 0.0), EPSILON);
                vec3 specular = numerator / denominator;
                float ndotl = max(dot(normal, lightDir), 0.0);
                radianceSum += (diffuseShare * albedo / PI + specular) * radiance * ndotl;
            }

            vec3 ambient = albedo * 0.03;
            vec3 color = ambient + radianceSum;
            color = color / (color + vec3(1.0));
            color = pow(color, vec3(1.0 / 2.2));
            output.color = vec4(color, 1.0);
            return output;
        }
    }
}

```

## Translate To Showcase Backends

This cell translates the same source file to graphics, compute-oriented, browser, and interchange targets. The assertions are deliberately concrete: if a backend regresses to malformed stage signatures or stale texture lowering, the notebook fails instead of displaying misleading output.

In [3]:
BACKENDS = {
    "metal": "cpp",
    "directx": "hlsl",
    "opengl": "glsl",
    "vulkan": "text",
    "cuda": "cuda",
    "hip": "cpp",
    "slang": "hlsl",
    "wgsl": "wgsl",
    "webgl": "glsl",
}

generated = {}
for backend in BACKENDS:
    generated[backend] = crosstl.translate(str(SOURCE_PATH), backend=backend)
    assert generated[backend].strip(), f"{backend} produced empty output"

metal_code = generated["metal"]
assert "vertex VertexOutput vertex_main" in metal_code
assert "fragment FragmentOutput" in metal_code
assert "texture2d<float> albedoMap" in metal_code
assert ".sample(" in metal_code
assert "gl_Position" not in metal_code

hlsl_code = generated["directx"]
assert "VertexOutput VSMain(VertexInput input)" in hlsl_code
assert "FragmentOutput PSMain(FragmentInput input)" in hlsl_code
assert "Texture2D albedoMap" in hlsl_code
assert ".Sample(" in hlsl_code

glsl_code = generated["opengl"]
assert "#version 450 core" in glsl_code
assert "uniform sampler2D albedoMap" in glsl_code
assert "texture(albedoMap" in glsl_code

slang_code = generated["slang"]
assert "Sampler2D<float4> albedoMap" in slang_code
assert "[shader(\"fragment\")]" in slang_code

wgsl_code = generated["wgsl"]
assert "@vertex" in wgsl_code
assert "@fragment" in wgsl_code

print("Translated and validated:")
for backend, code in generated.items():
    print(f"{backend:<8} {len(code.splitlines()):>5} lines")


Translated and validated:
metal      162 lines
directx    139 lines
opengl     137 lines
vulkan     665 lines
cuda      3625 lines
hip       3610 lines
slang      137 lines
wgsl       147 lines
webgl      138 lines


## Metal Output

The Metal output should have concrete stage signatures, `[[stage_in]]` inputs, `[[position]]`/`[[color(0)]]` outputs, and native texture sampling. This is the path that exposed the stale notebook output before.

In [4]:
display(Markdown(f"""```cpp
{generated['metal']}
```"""))


```cpp

#include <metal_stdlib>
using namespace metal;

__attribute__((unused)) constant float PI = 3.14159265359;
__attribute__((unused)) constant float EPSILON = 0.0001;
__attribute__((unused)) constant int MAX_LIGHTS = 4;

struct VertexInput {
  float3 position [[attribute(0)]];
  float3 normal [[attribute(1)]];
  float3 tangent [[attribute(2)]];
  float2 texCoord [[attribute(3)]];
};
struct VertexOutput {
  float3 worldPosition;
  float3 worldNormal;
  float3 worldTangent;
  float3 worldBitangent;
  float2 uv;
  float4 position [[position]];
};
struct FragmentInput {
  float3 worldPosition;
  float3 worldNormal;
  float3 worldTangent;
  float3 worldBitangent;
  float2 uv;
};
struct FragmentOutput {
  float4 color [[color(0)]];
};
float distributionGGX(float3 normal, float3 halfVector, float roughness) {
  float alpha = roughness * roughness;
  float alpha2 = alpha * alpha;
  float ndoth = max(dot(normal, halfVector), 0.0);
  float denom = ndoth * ndoth * (alpha2 - 1.0) + 1.0;
  return alpha2 / max(PI * denom * denom, EPSILON);
}

float geometrySchlickGGX(float ndotv, float roughness) {
  float k = (roughness + 1.0) * (roughness + 1.0) / 8.0;
  return ndotv / max(ndotv * (1.0 - k) + k, EPSILON);
}

float geometrySmith(float3 normal, float3 viewDir, float3 lightDir,
                    float roughness) {
  float ndotv = max(dot(normal, viewDir), 0.0);
  float ndotl = max(dot(normal, lightDir), 0.0);
  return geometrySchlickGGX(ndotv, roughness) *
         geometrySchlickGGX(ndotl, roughness);
}

float3 fresnelSchlick(float cosTheta, float3 baseReflectance) {
  return baseReflectance +
         (float3(1.0) - baseReflectance) * pow(max(1.0 - cosTheta, 0.0), 5.0);
}

// Vertex Shader
vertex VertexOutput vertex_main(VertexInput input [[stage_in]],
                                constant float4x4 &modelMatrix [[buffer(0)]],
                                constant float4x4 &viewProjectionMatrix
                                [[buffer(1)]],
                                constant float3x3 &normalMatrix [[buffer(2)]],
                                constant float3 &cameraPosition [[buffer(3)]],
                                constant float3 *lightPositions [[buffer(4)]],
                                constant float3 *lightColors [[buffer(5)]],
                                constant int &activeLightCount [[buffer(6)]],
                                constant float3 &baseColor [[buffer(7)]],
                                constant float &metallicFactor [[buffer(8)]],
                                constant float &roughnessFactor [[buffer(9)]],
                                texture2d<float> albedoMap [[texture(0)]],
                                texture2d<float> normalMap [[texture(1)]],
                                texture2d<float> metallicRoughnessMap
                                [[texture(2)]]) {
  VertexOutput output;
  float4 worldPosition = modelMatrix * float4(input.position, 1.0);
  output.worldPosition = worldPosition.xyz;
  output.worldNormal = normalize(normalMatrix * input.normal);
  output.worldTangent = normalize(normalMatrix * input.tangent);
  output.worldBitangent =
      normalize(cross(output.worldNormal, output.worldTangent));
  output.uv = input.texCoord;
  output.position = viewProjectionMatrix * worldPosition;
  return output;
}

float3 decodeNormal(float3 encodedNormal, float3x3 tangentFrame) {
  float3 tangentNormal = encodedNormal * float3(2.0) - float3(1.0);
  return normalize(tangentFrame * tangentNormal);
}

// Fragment Shader
fragment FragmentOutput
fragment_main(FragmentInput input [[stage_in]],
              constant float4x4 &modelMatrix [[buffer(0)]],
              constant float4x4 &viewProjectionMatrix [[buffer(1)]],
              constant float3x3 &normalMatrix [[buffer(2)]],
              constant float3 &cameraPosition [[buffer(3)]],
              constant float3 *lightPositions [[buffer(4)]],
              constant float3 *lightColors [[buffer(5)]],
              constant int &activeLightCount [[buffer(6)]],
              constant float3 &baseColor [[buffer(7)]],
              constant float &metallicFactor [[buffer(8)]],
              constant float &roughnessFactor [[buffer(9)]],
              texture2d<float> albedoMap [[texture(0)]],
              texture2d<float> normalMap [[texture(1)]],
              texture2d<float> metallicRoughnessMap [[texture(2)]]) {
  FragmentOutput output;
  float3x3 tangentFrame =
      float3x3(normalize(input.worldTangent), normalize(input.worldBitangent),
               normalize(input.worldNormal));
  float3 normal = decodeNormal(
      normalMap
          .sample(sampler(mag_filter::linear, min_filter::linear), input.uv)
          .rgb,
      tangentFrame);
  float3 viewDir = normalize(cameraPosition - input.worldPosition);
  float3 albedo =
      albedoMap
          .sample(sampler(mag_filter::linear, min_filter::linear), input.uv)
          .rgb *
      baseColor;
  float3 materialSample =
      metallicRoughnessMap
          .sample(sampler(mag_filter::linear, min_filter::linear), input.uv)
          .rgb;
  float metallic = clamp(materialSample.b * metallicFactor, 0.0, 1.0);
  float roughness = clamp(materialSample.g * roughnessFactor, 0.04, 1.0);
  float3 baseReflectance = mix(float3(0.04), albedo, metallic);
  float3 radianceSum = float3(0.0);
  for (int i = 0; i < MAX_LIGHTS; ++i) {
    if (i >= activeLightCount) {
      break;
    }
    float3 lightVector = lightPositions[i] - input.worldPosition;
    float distanceSq = max(dot(lightVector, lightVector), EPSILON);
    float3 lightDir = normalize(lightVector);
    float3 halfVector = normalize(viewDir + lightDir);
    float attenuation = 1.0 / distanceSq;
    float3 radiance = lightColors[i] * float3(attenuation);
    float normalDistribution = distributionGGX(normal, halfVector, roughness);
    float geometry = geometrySmith(normal, viewDir, lightDir, roughness);
    float3 fresnel =
        fresnelSchlick(max(dot(halfVector, viewDir), 0.0), baseReflectance);
    float3 diffuseShare = (float3(1.0) - fresnel) * float3((1.0 - metallic));
    float3 numerator = float3(normalDistribution * geometry) * fresnel;
    float denominator = max(4.0 * max(dot(normal, viewDir), 0.0) *
                                max(dot(normal, lightDir), 0.0),
                            EPSILON);
    float3 specular = numerator / float3(denominator);
    float ndotl = max(dot(normal, lightDir), 0.0);
    radianceSum +=
        (diffuseShare * albedo / PI + specular) * radiance * float3(ndotl);
  }
  float3 ambient = albedo * float3(0.03);
  float3 color = ambient + radianceSum;
  color = color / (color + float3(1.0));
  color = pow(color, float3(1.0 / 2.2));
  output.color = float4(color, 1.0);
  return output;
}

```

## DirectX And OpenGL Output

These two outputs show the same shader lowering to HLSL semantics and GLSL stage guards.

In [5]:
display(Markdown(f"""### DirectX / HLSL
```hlsl
{generated['directx']}
```"""))
display(Markdown(f"""### OpenGL / GLSL
```glsl
{generated['opengl']}
```"""))


### DirectX / HLSL
```hlsl

static const float PI = 3.14159265359;
static const float EPSILON = 0.0001;
static const int MAX_LIGHTS = 4;

struct VertexInput
{
    float3 position : POSITION;
    float3 normal : TEXCOORD0;
    float3 tangent : TEXCOORD1;
    float2 texCoord : TEXCOORD2;
};
struct VertexOutput
{
    float3 worldPosition : TEXCOORD0;
    float3 worldNormal : TEXCOORD1;
    float3 worldTangent : TEXCOORD2;
    float3 worldBitangent : TEXCOORD3;
    float2 uv : TEXCOORD4;
    float4 position : SV_Position;
};
struct FragmentInput
{
    float3 worldPosition : TEXCOORD0;
    float3 worldNormal : TEXCOORD1;
    float3 worldTangent : TEXCOORD2;
    float3 worldBitangent : TEXCOORD3;
    float2 uv : TEXCOORD4;
};
struct FragmentOutput
{
    float4 color : SV_TARGET;
};
float4x4 modelMatrix;
float4x4 viewProjectionMatrix;
float3x3 normalMatrix;
Texture2D albedoMap : register(t0);
SamplerState albedoMapSampler : register(s0);
Texture2D normalMap : register(t1);
SamplerState normalMapSampler : register(s1);
Texture2D metallicRoughnessMap : register(t2);
SamplerState metallicRoughnessMapSampler : register(s2);
float3 cameraPosition;
float3 lightPositions[MAX_LIGHTS];
float3 lightColors[MAX_LIGHTS];
int activeLightCount;
float3 baseColor;
float metallicFactor;
float roughnessFactor;
float distributionGGX(float3 normal, float3 halfVector, float roughness)
{
    float alpha = (roughness * roughness);
    float alpha2 = (alpha * alpha);
    float ndoth = max(dot(normal, halfVector), 0.0);
    float denom = (((ndoth * ndoth) * (alpha2 - 1.0)) + 1.0);
    return (alpha2 / max(((PI * denom) * denom), EPSILON));
}

float geometrySchlickGGX(float ndotv, float roughness)
{
    float k = (((roughness + 1.0) * (roughness + 1.0)) / 8.0);
    return (ndotv / max(((ndotv * (1.0 - k)) + k), EPSILON));
}

float geometrySmith(float3 normal, float3 viewDir, float3 lightDir, float roughness)
{
    float ndotv = max(dot(normal, viewDir), 0.0);
    float ndotl = max(dot(normal, lightDir), 0.0);
    return (geometrySchlickGGX(ndotv, roughness) * geometrySchlickGGX(ndotl, roughness));
}

float3 fresnelSchlick(float cosTheta, float3 baseReflectance)
{
    return (baseReflectance + ((float3(1.0, 1.0, 1.0) - baseReflectance) * pow(max((1.0 - cosTheta), 0.0), 5.0)));
}

// Vertex Shader
VertexOutput VSMain(VertexInput input)
{
    VertexOutput output;
    float4 worldPosition = mul(modelMatrix, float4(input.position, 1.0));
    output.worldPosition = worldPosition.xyz;
    output.worldNormal = normalize(mul(normalMatrix, input.normal));
    output.worldTangent = normalize(mul(normalMatrix, input.tangent));
    output.worldBitangent = normalize(cross(output.worldNormal, output.worldTangent));
    output.uv = input.texCoord;
    output.position = mul(viewProjectionMatrix, worldPosition);
    return output;
}

float3 decodeNormal(float3 encodedNormal, float3x3 tangentFrame)
{
    float3 tangentNormal = ((encodedNormal * 2.0) - 1.0);
    return normalize(mul(tangentFrame, tangentNormal));
}

// Fragment Shader
FragmentOutput PSMain(FragmentInput input)
{
    FragmentOutput output;
    float3x3 tangentFrame =
        float3x3(normalize(input.worldTangent), normalize(input.worldBitangent), normalize(input.worldNormal));
    float3 normal = decodeNormal(normalMap.Sample(normalMapSampler, input.uv).rgb, tangentFrame);
    float3 viewDir = normalize((cameraPosition - input.worldPosition));
    float3 albedo = (albedoMap.Sample(albedoMapSampler, input.uv).rgb * baseColor);
    float3 materialSample = metallicRoughnessMap.Sample(metallicRoughnessMapSampler, input.uv).rgb;
    float metallic = clamp((materialSample.b * metallicFactor), 0.0, 1.0);
    float roughness = clamp((materialSample.g * roughnessFactor), 0.04, 1.0);
    float3 baseReflectance = lerp(float3(0.04, 0.04, 0.04), albedo, metallic);
    float3 radianceSum = float3(0.0, 0.0, 0.0);
    for (int i = 0; (i < MAX_LIGHTS); ++i)
    {
        if ((i >= activeLightCount))
        {
            break;
        }
        float3 lightVector = (lightPositions[i] - input.worldPosition);
        float distanceSq = max(dot(lightVector, lightVector), EPSILON);
        float3 lightDir = normalize(lightVector);
        float3 halfVector = normalize((viewDir + lightDir));
        float attenuation = (1.0 / distanceSq);
        float3 radiance = (lightColors[i] * attenuation);
        float normalDistribution = distributionGGX(normal, halfVector, roughness);
        float geometry = geometrySmith(normal, viewDir, lightDir, roughness);
        float3 fresnel = fresnelSchlick(max(dot(halfVector, viewDir), 0.0), baseReflectance);
        float3 diffuseShare = ((float3(1.0, 1.0, 1.0) - fresnel) * (1.0 - metallic));
        float3 numerator = ((normalDistribution * geometry) * fresnel);
        float denominator = max(((4.0 * max(dot(normal, viewDir), 0.0)) * max(dot(normal, lightDir), 0.0)), EPSILON);
        float3 specular = (numerator / denominator);
        float ndotl = max(dot(normal, lightDir), 0.0);
        radianceSum += (((((diffuseShare * albedo) / PI) + specular) * radiance) * ndotl);
    }
    float3 ambient = (albedo * 0.03);
    float3 color = (ambient + radianceSum);
    color = (color / (color + float3(1.0, 1.0, 1.0)));
    color = pow(color, float3((1.0 / 2.2), (1.0 / 2.2), (1.0 / 2.2)));
    output.color = float4(color, 1.0);
    return output;
}

```

### OpenGL / GLSL
```glsl
#version 450 core
const float PI = 3.14159265359;
const float EPSILON = 0.0001;
const int MAX_LIGHTS = 4;

#ifdef GL_VERTEX_SHADER
in vec3 position;
in vec3 normal;
in vec3 tangent;
in vec2 texCoord;
#endif
#ifdef GL_VERTEX_SHADER
out vec3 worldPosition;
out vec3 worldNormal;
out vec3 worldTangent;
out vec3 worldBitangent;
out vec2 uv;
out vec4 out_position;
#endif
#ifdef GL_FRAGMENT_SHADER
in vec3 in_worldPosition;
in vec3 in_worldNormal;
in vec3 in_worldTangent;
in vec3 in_worldBitangent;
in vec2 in_uv;
#endif
#ifdef GL_FRAGMENT_SHADER
layout(location = 0) out vec4 out_color;
#endif
struct FragmentOutput {
  vec4 color;
};

uniform mat4 modelMatrix;
uniform mat4 viewProjectionMatrix;
uniform mat3 normalMatrix;
layout(binding = 0) uniform sampler2D albedoMap;
layout(binding = 1) uniform sampler2D normalMap;
layout(binding = 2) uniform sampler2D metallicRoughnessMap;
uniform vec3 cameraPosition;
uniform vec3 lightPositions[MAX_LIGHTS];
uniform vec3 lightColors[MAX_LIGHTS];
uniform int activeLightCount;
uniform vec3 baseColor;
uniform float metallicFactor;
uniform float roughnessFactor;
float distributionGGX(vec3 normal, vec3 halfVector, float roughness) {
  float alpha = (roughness * roughness);
  float alpha2 = (alpha * alpha);
  float ndoth = max(dot(normal, halfVector), 0.0);
  float denom = (((ndoth * ndoth) * (alpha2 - 1.0)) + 1.0);
  return (alpha2 / max(((PI * denom) * denom), EPSILON));
}

float geometrySchlickGGX(float ndotv, float roughness) {
  float k = (((roughness + 1.0) * (roughness + 1.0)) / 8.0);
  return (ndotv / max(((ndotv * (1.0 - k)) + k), EPSILON));
}

float geometrySmith(vec3 normal, vec3 viewDir, vec3 lightDir, float roughness) {
  float ndotv = max(dot(normal, viewDir), 0.0);
  float ndotl = max(dot(normal, lightDir), 0.0);
  return (geometrySchlickGGX(ndotv, roughness) *
          geometrySchlickGGX(ndotl, roughness));
}

vec3 fresnelSchlick(float cosTheta, vec3 baseReflectance) {
  return (baseReflectance + ((vec3(1.0) - baseReflectance) *
                             pow(max((1.0 - cosTheta), 0.0), 5.0)));
}

#ifdef GL_VERTEX_SHADER
// Vertex Shader
void main() {
  vec4 worldPosition_ = (modelMatrix * vec4(position, 1.0));
  worldPosition = worldPosition_.xyz;
  worldNormal = normalize((normalMatrix * normal));
  worldTangent = normalize((normalMatrix * tangent));
  worldBitangent = normalize(cross(worldNormal, worldTangent));
  uv = texCoord;
  out_position = (viewProjectionMatrix * worldPosition_);
  return;
}

#endif
#ifdef GL_FRAGMENT_SHADER
// Fragment Shader
vec3 decodeNormal(vec3 encodedNormal, mat3 tangentFrame) {
  vec3 tangentNormal = ((encodedNormal * 2.0) - 1.0);
  return normalize((tangentFrame * tangentNormal));
}

void main() {
  mat3 tangentFrame =
      mat3(normalize(in_worldTangent), normalize(in_worldBitangent),
           normalize(in_worldNormal));
  vec3 normal = decodeNormal(texture(normalMap, in_uv).rgb, tangentFrame);
  vec3 viewDir = normalize((cameraPosition - in_worldPosition));
  vec3 albedo = (texture(albedoMap, in_uv).rgb * baseColor);
  vec3 materialSample = texture(metallicRoughnessMap, in_uv).rgb;
  float metallic = clamp((materialSample.b * metallicFactor), 0.0, 1.0);
  float roughness = clamp((materialSample.g * roughnessFactor), 0.04, 1.0);
  vec3 baseReflectance = mix(vec3(0.04), albedo, metallic);
  vec3 radianceSum = vec3(0.0);
  for (int i = 0; (i < MAX_LIGHTS); (++i)) {
    if ((i >= activeLightCount)) {
      break;
    }
    vec3 lightVector = (lightPositions[i] - in_worldPosition);
    float distanceSq = max(dot(lightVector, lightVector), EPSILON);
    vec3 lightDir = normalize(lightVector);
    vec3 halfVector = normalize((viewDir + lightDir));
    float attenuation = (1.0 / distanceSq);
    vec3 radiance = (lightColors[i] * attenuation);
    float normalDistribution = distributionGGX(normal, halfVector, roughness);
    float geometry = geometrySmith(normal, viewDir, lightDir, roughness);
    vec3 fresnel =
        fresnelSchlick(max(dot(halfVector, viewDir), 0.0), baseReflectance);
    vec3 diffuseShare = ((vec3(1.0) - fresnel) * (1.0 - metallic));
    vec3 numerator = ((normalDistribution * geometry) * fresnel);
    float denominator = max(((4.0 * max(dot(normal, viewDir), 0.0)) *
                             max(dot(normal, lightDir), 0.0)),
                            EPSILON);
    vec3 specular = (numerator / denominator);
    float ndotl = max(dot(normal, lightDir), 0.0);
    radianceSum +=
        (((((diffuseShare * albedo) / PI) + specular) * radiance) * ndotl);
  }
  vec3 ambient = (albedo * 0.03);
  vec3 color = (ambient + radianceSum);
  color = (color / (color + vec3(1.0)));
  color = pow(color, vec3((1.0 / 2.2)));
  out_color = vec4(color, 1.0);
  return;
}

#endif

```

## Full Backend Matrix

The remaining targets are summarized with short previews to keep the notebook readable while still proving that each translation path produces output from the same source.

In [6]:
def preview(code: str, max_lines: int = 36) -> str:
    lines = code.splitlines()
    head = "\n".join(lines[:max_lines])
    if len(lines) > max_lines:
        return f"{head}\n// ... ({len(lines)} total lines)"
    return head

rows = ["| Backend | Lines | Key signal |", "|---|---:|---|"]
signals = {
    "metal": "vertex/fragment Metal stage functions",
    "directx": "VSMain/PSMain with HLSL semantics",
    "opengl": "GLSL 450 stage guards",
    "vulkan": "SPIR-V assembly entry points",
    "cuda": "CUDA helper and kernel-oriented output",
    "hip": "HIP helper and kernel-oriented output",
    "slang": "Slang shader attributes and samplers",
    "wgsl": "WGSL vertex/fragment entry points",
    "webgl": "GLSL ES/WebGL-compatible output",
}
for backend, code in generated.items():
    rows.append(f"| `{backend}` | {len(code.splitlines())} | {signals[backend]} |")

display(Markdown("\n".join(rows)))

for backend in ["vulkan", "slang", "wgsl", "webgl", "cuda", "hip"]:
    language = BACKENDS[backend]
    display(Markdown(f"""### {backend}
```{language}
{preview(generated[backend])}
```"""))


| Backend | Lines | Key signal |
|---|---:|---|
| `metal` | 162 | vertex/fragment Metal stage functions |
| `directx` | 139 | VSMain/PSMain with HLSL semantics |
| `opengl` | 137 | GLSL 450 stage guards |
| `vulkan` | 665 | SPIR-V assembly entry points |
| `cuda` | 3625 | CUDA helper and kernel-oriented output |
| `hip` | 3610 | HIP helper and kernel-oriented output |
| `slang` | 137 | Slang shader attributes and samplers |
| `wgsl` | 147 | WGSL vertex/fragment entry points |
| `webgl` | 138 | GLSL ES/WebGL-compatible output |

### vulkan
```text
; SPIR-V
; Version: 1.0
; Generator: Khronos SPIR-V Tools Assembler; 0
; Bound: 452
; Schema: 0
               OpCapability Shader
          %1 = OpExtInstImport "GLSL.std.450"
               OpMemoryModel Logical GLSL450
               OpEntryPoint Vertex %2 "main" %CrossGL_vertex_output_main_worldPosition %CrossGL_vertex_output_main_worldNormal %CrossGL_vertex_output_main_worldTangent %CrossGL_vertex_output_main_worldBitangent %CrossGL_vertex_output_main_uv %gl_Position %CrossGL_vertex_input_input_position %CrossGL_vertex_input_input_normal %CrossGL_vertex_input_input_tangent %CrossGL_vertex_input_input_texCoord
               OpEntryPoint Fragment %13 "main" %CrossGL_fragment_output_main_color %CrossGL_fragment_input_input_worldPosition %CrossGL_fragment_input_input_worldNormal %CrossGL_fragment_input_input_worldTangent %CrossGL_fragment_input_input_worldBitangent %CrossGL_fragment_input_input_uv
               OpExecutionMode %13 OriginUpperLeft
               OpName %VertexInput "VertexInput"
               OpMemberName %VertexInput 0 "position"
               OpMemberName %VertexInput 1 "normal"
               OpMemberName %VertexInput 2 "tangent"
               OpMemberName %VertexInput 3 "texCoord"
               OpName %VertexOutput "VertexOutput"
               OpMemberName %VertexOutput 0 "worldPosition"
               OpMemberName %VertexOutput 1 "worldNormal"
               OpMemberName %VertexOutput 2 "worldTangent"
               OpMemberName %VertexOutput 3 "worldBitangent"
               OpMemberName %VertexOutput 4 "uv"
               OpMemberName %VertexOutput 5 "position"
               OpName %FragmentInput "FragmentInput"
               OpMemberName %FragmentInput 0 "worldPosition"
               OpMemberName %FragmentInput 1 "worldNormal"
               OpMemberName %FragmentInput 2 "worldTangent"
               OpMemberName %FragmentInput 3 "worldBitangent"
               OpMemberName %FragmentInput 4 "uv"
               OpName %FragmentOutput "FragmentOutput"
               OpMemberName %FragmentOutput 0 "color"
               OpName %PI "PI"
               OpName %EPSILON "EPSILON"
               OpName %MAX_LIGHTS "MAX_LIGHTS"
               OpName %normal "normal"
               OpName %halfVector "halfVector"
// ... (665 total lines)
```

### slang
```hlsl
static const float PI = 3.14159265359;
static const float EPSILON = 0.0001;
static const int MAX_LIGHTS = 4;

struct VertexInput
{
    float3 position : POSITION;
    float3 normal : TEXCOORD0;
    float3 tangent : TEXCOORD1;
    float2 texCoord : TEXCOORD2;
};

struct VertexOutput
{
    float3 worldPosition : TEXCOORD0;
    float3 worldNormal : TEXCOORD1;
    float3 worldTangent : TEXCOORD2;
    float3 worldBitangent : TEXCOORD3;
    float2 uv : TEXCOORD4;
    float4 position : SV_Position;
};

struct FragmentInput
{
    float3 worldPosition : TEXCOORD0;
    float3 worldNormal : TEXCOORD1;
    float3 worldTangent : TEXCOORD2;
    float3 worldBitangent : TEXCOORD3;
    float2 uv : TEXCOORD4;
};

struct FragmentOutput
{
    float4 color : SV_Target;
};

// ... (137 total lines)
```

### wgsl
```wgsl
// Generated by CrossGL for WebGPU WGSL

struct VertexInput {
    @location(0) position: vec3<f32>,
    @location(1) normal: vec3<f32>,
    @location(2) tangent: vec3<f32>,
    @location(3) texCoord: vec2<f32>,
};

struct VertexOutput {
    @location(0) worldPosition: vec3<f32>,
    @location(1) worldNormal: vec3<f32>,
    @location(2) worldTangent: vec3<f32>,
    @location(3) worldBitangent: vec3<f32>,
    @location(4) uv: vec2<f32>,
    @builtin(position) position: vec4<f32>,
};

struct FragmentInput {
    @location(0) worldPosition: vec3<f32>,
    @location(1) worldNormal: vec3<f32>,
    @location(2) worldTangent: vec3<f32>,
    @location(3) worldBitangent: vec3<f32>,
    @location(4) uv: vec2<f32>,
};

struct FragmentOutput {
    @location(0) color: vec4<f32>,
};

const PI: f32 = 3.14159265359;
const EPSILON: f32 = 0.0001;
const MAX_LIGHTS: i32 = 4;

@group(0) @binding(0)
var<uniform> modelMatrix: mat4x4<f32>;
// ... (147 total lines)
```

### webgl
```glsl
#version 300 es
precision highp float;
precision highp int;
const float PI = 3.14159265359;
const float EPSILON = 0.0001;
const int MAX_LIGHTS = 4;

#ifdef GL_VERTEX_SHADER
in vec3 position;
in vec3 normal;
in vec3 tangent;
in vec2 texCoord;
#endif
#ifdef GL_VERTEX_SHADER
out vec3 worldPosition;
out vec3 worldNormal;
out vec3 worldTangent;
out vec3 worldBitangent;
out vec2 uv;
out vec4 out_position;
#endif
#ifdef GL_FRAGMENT_SHADER
in vec3 worldPosition;
in vec3 worldNormal;
in vec3 worldTangent;
in vec3 worldBitangent;
in vec2 uv;
#endif
#ifdef GL_FRAGMENT_SHADER
layout(location = 0) out vec4 out_color;
#endif
struct FragmentOutput {
  vec4 color;
};

uniform mat4 modelMatrix;
// ... (138 total lines)
```

### cuda
```cuda
#include <cuda_fp16.h>
#include <cuda_runtime.h>
#include <device_launch_parameters.h>
__device__ inline float cgl_float3_dot(float3 lhs, float3 rhs) {
  return (lhs.x * rhs.x) + (lhs.y * rhs.y) + (lhs.z * rhs.z);
}

__device__ inline float3 cgl_float3_sub(float3 lhs, float3 rhs) {
  return make_float3((lhs.x - rhs.x), (lhs.y - rhs.y), (lhs.z - rhs.z));
}

__device__ inline float3 cgl_float3_mul_scalar(float3 lhs, float rhs) {
  return make_float3((lhs.x * rhs), (lhs.y * rhs), (lhs.z * rhs));
}

__device__ inline float3 cgl_float3_add(float3 lhs, float3 rhs) {
  return make_float3((lhs.x + rhs.x), (lhs.y + rhs.y), (lhs.z + rhs.z));
}

__device__ inline float cgl_float3_length(float3 value) {
  return sqrtf((value.x * value.x) + (value.y * value.y) + (value.z * value.z));
}

__device__ inline float3 cgl_float3_normalize(float3 value) {
  float inv_length = 1.0f / cgl_float3_length(value);
  return make_float3((value.x * inv_length), (value.y * inv_length),
                     (value.z * inv_length));
}

__device__ inline float3 cgl_float3_cross(float3 lhs, float3 rhs) {
  return make_float3((lhs.y * rhs.z - lhs.z * rhs.y),
                     (lhs.z * rhs.x - lhs.x * rhs.z),
                     (lhs.x * rhs.y - lhs.y * rhs.x));
}

__device__ inline float3 cgl_float3_sub_scalar(float3 lhs, float rhs) {
// ... (3625 total lines)
```

### hip
```cpp
#include <hip/device_functions.h>
#include <hip/hip_fp16.h>
#include <hip/hip_runtime.h>
#include <hip/hip_runtime_api.h>
#include <hip/math_functions.h>
__device__ inline float cgl_float3_dot(float3 lhs, float3 rhs) {
  return (lhs.x * rhs.x) + (lhs.y * rhs.y) + (lhs.z * rhs.z);
}

__device__ inline float3 cgl_float3_sub(float3 lhs, float3 rhs) {
  return make_float3((lhs.x - rhs.x), (lhs.y - rhs.y), (lhs.z - rhs.z));
}

__device__ inline float3 cgl_float3_mul_scalar(float3 lhs, float rhs) {
  return make_float3((lhs.x * rhs), (lhs.y * rhs), (lhs.z * rhs));
}

__device__ inline float3 cgl_float3_add(float3 lhs, float3 rhs) {
  return make_float3((lhs.x + rhs.x), (lhs.y + rhs.y), (lhs.z + rhs.z));
}

__device__ inline float cgl_float3_length(float3 value) {
  return sqrtf((value.x * value.x) + (value.y * value.y) + (value.z * value.z));
}

__device__ inline float3 cgl_float3_normalize(float3 value) {
  float inv_length = 1.0f / cgl_float3_length(value);
  return make_float3((value.x * inv_length), (value.y * inv_length),
                     (value.z * inv_length));
}

__device__ inline float3 cgl_float3_cross(float3 lhs, float3 rhs) {
  return make_float3((lhs.y * rhs.z - lhs.z * rhs.y),
                     (lhs.z * rhs.x - lhs.x * rhs.z),
                     (lhs.x * rhs.y - lhs.y * rhs.x));
}
// ... (3610 total lines)
```

## What This Demonstrates

One CrossGL source drives all showcased backends. The notebook validates the important output invariants before displaying them, so it is safe to use as a quick smoke test for README-facing examples.